# LLM Sampling Strategies: Temperature, Top-K & Top-P

## 1. Logits and Probabilities
- **Logits:** Raw scores produced by the model for every vocabulary token.
- **Softmax:** Converts logits into probabilities.

\[
P(x_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
\]

## 2. Greedy Decoding
Always selects the token with the highest probability.

\[
x_{next}=\arg\max_i P(x_i)
\]

- Deterministic.
- Predictable output.
- Can cause repetitive generation.

## 3. Temperature
Controls the randomness of token selection by changing the probability distribution.

\[
\boxed{P_i=\frac{\exp(z_i/T)}{\sum_j\exp(z_j/T)}}
\]

| Temperature | Behavior |
|---|---|
| T → 0 | Greedy-like selection |
| 0.2–0.5 | More focused |
| 0.7–1.0 | Balanced |
| Above 1.0 | More diverse/random |

- Low temperature: Sharpens probability distribution.
- High temperature: Flattens probability distribution.
- Temperature = 1: Original softmax distribution.

## 4. Top-K Sampling
Keeps only the K most probable tokens and removes the remaining candidates.

Example: K = 3

Original probabilities:
- A = 45%
- B = 25%
- C = 15%
- D = 10%
- E = 5%

After Top-K filtering, only A, B and C remain. Their probabilities are renormalized.

**Key idea:** Fixed number of candidate tokens.

## 5. Top-P Sampling (Nucleus Sampling)
Keeps the smallest set of highest-probability tokens whose cumulative probability reaches or exceeds P.

Example: P = 0.80

- A = 25%
- B = 22%
- C = 20%
- D = 18%
- E = 15%

Cumulative probabilities:
- A = 25%
- A + B = 47%
- A + B + C = 67%
- A + B + C + D = 85%

Therefore, A, B, C and D are retained.

**Key idea:** Dynamic number of candidate tokens based on cumulative probability.

## 6. Top-K vs Top-P vs Temperature

| Technique | Purpose |
|---|---|
| Temperature | Controls probability distribution sharpness |
| Top-K | Retains fixed K highest-probability candidates |
| Top-P | Retains candidates based on cumulative probability |
| Greedy | Always chooses the highest-probability token |

## 7. Repetition Penalty
Discourages the model from repeatedly generating tokens that have already appeared.

In Hugging Face:

```python
repetition_penalty=1.1
```

Values above 1 penalize repeated tokens. Excessive penalties can make output unnatural.

## 8. Hugging Face Implementation

```python
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1
)
```

Note: `do_sample=True` enables sampling. Without sampling, temperature and Top-K/Top-P sampling controls do not operate as intended.

## 9. Important Takeaways

1. Logits are raw scores; softmax converts them into probabilities.
2. Greedy decoding always selects the highest-probability token.
3. Temperature controls randomness.
4. Top-K limits the number of candidate tokens.
5. Top-P dynamically selects candidates using cumulative probability.
6. Repetition penalty discourages repetitive output.

**Remember:**
- Temperature = How random?
- Top-K = How many candidates?
- Top-P = How much cumulative probability?
- Greedy = Always choose the best-scoring token.

In [3]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name)

model.eval()

prompt = "The future of artificial intelligence"

inputs = tokenizer(prompt, return_tensors="pt")

experiments = [
    {
        "name": "Greedy",
        "do_sample": False
    },
    {
        "name": "Low Temperature",
        "do_sample": True,
        "temperature": 0.3,
        "top_k": 50
    },
    {
        "name": "High Temperature",
        "do_sample": True,
        "temperature": 1.5,
        "top_k": 50
    },
    {
        "name": "Top-P Sampling",
        "do_sample": True,
        "temperature": 0.8,
        "top_p": 0.9
    }
]

for experiment in experiments:

    name = experiment.pop("name")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            **experiment
        )

    print("\n", "=" * 50)
    print(name)
    print(tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ))


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


Greedy
The future of artificial intelligence is not yet clear.












































































Low Temperature
The future of artificial intelligence, and its future, is in the hands of the people of the world.

High Temperature
The future of artificial intelligence, like Siri, should come with some caveats. But one important consideration for many observers, though, remains that the future is far different after all. An artificial intelligence can tell you which type of product is being placed for you to work for, while an artificial program may tell an interviewer who is not on campus where certain kinds of jobs aren't available, and may need to wait a while for the

Top-P Sampling
The future of artificial intelligence could be far from clear.
